## 1. Bibliotecas e Dados

In [71]:
!uv pip install "numpy==1.26.4" "spacy==3.7.4" scispacy --force-reinstall
!uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

Using Python 3.11.13 environment at: /usr
Resolved 46 packages in 380ms
Prepared 46 packages in 7ms
Uninstalled 46 packages in 612ms
Installed 46 packages in 286ms
 ~ annotated-types==0.8.0
 ~ blis==0.7.11
 ~ catalogue==2.0.10
 ~ certifi==2026.7.22
 ~ charset-normalizer==3.5.1
 ~ click==8.5.0
 ~ cloudpathlib==0.16.0
 ~ cloudpickle==3.1.2
 ~ confection==0.1.5
 ~ conllu==6.0.0
 ~ cymem==2.0.13
 ~ idna==3.19
 ~ jinja2==3.1.6
 ~ joblib==1.6.0
 ~ langcodes==3.5.1
 ~ markupsafe==3.0.3
 ~ murmurhash==1.0.15
 ~ narwhals==2.26.0
 ~ nmslib-metabrainz==2.1.3
 ~ numpy==1.26.4
 ~ packaging==26.3
 ~ preshed==3.0.13
 ~ psutil==7.2.2
 ~ pybind11==3.1.0
 ~ pydantic==2.13.5
 ~ pydantic-core==2.46.5
 ~ pysbd==0.3.4
 ~ requests==2.34.2
 ~ scikit-learn==1.9.0
 ~ scipy==1.17.1
 ~ scispacy==0.6.2
 ~ setuptools==84.0.0
 ~ smart-open==6.4.0
 ~ spacy==3.7.4
 ~ spacy-legacy==3.0.12
 ~ spacy-loggers==1.0.5
 ~ srsly==2.5.3
 ~ thinc==8.2.5
 ~ threadpoolctl==3.6.0
 ~ tqdm==4.70.0
 ~ typer==0.9.4
 ~ typing-extensions

In [72]:
import spacy
import pandas as pd
import re
from pathlib import Path
from spacy.matcher import Matcher
from spacy.util import filter_spans


ROOT = Path('')
data = pd.read_csv(ROOT / 'cases.csv')
metadata = pd.read_csv(ROOT / 'metadata.csv')

In [73]:
# merge e seleção do text
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text', 'gender', 'case_id', 'major_mesh_terms', 'mesh_terms']]
text = full_data['case_text'][0]

# modelo médico scispaCy
nlp_med = spacy.load('en_ner_bc5cdr_md')
doc = nlp_med(text)

## 2. Captura de Medidas (Regex)

In [74]:
measurement_pattern = re.compile(r'(\d+(?:,\d+)?(?:\.\d+)?)\s*(cm|mm|ng/ml|iu/ml|mg)')
measurements = []

for match in measurement_pattern.finditer(text):
    value, unit = match.group(1), match.group(2)
    measurements.append({
        'node_type': 'ExamResult',
        'label_original': match.group(0),
        'label_normalizado': f"{value} {unit}",
        'token_start': -1,
        'token_end': -1,
        'span_start': match.start(),
        'span_end': match.end(),
        'value': value,
        'unit': unit
    })

measurements_df = pd.DataFrame(measurements)
display(measurements_df.head())

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,ExamResult,6cm,6 cm,-1,-1,337,340,6,cm
1,ExamResult,6cm,6 cm,-1,-1,516,519,6,cm
2,ExamResult,9cm,9 cm,-1,-1,522,525,9,cm
3,ExamResult,"12,476.5ng/ml","12,476.5 ng/ml",-1,-1,777,790,"12,476.5",ng/ml
4,ExamResult,6iu/ml,6 iu/ml,-1,-1,837,843,6,iu/ml


## 3. Extração de Entidades (scispaCy + Matcher) e Unificação de Nós

### Identificar paciente

In [75]:
extracted_entities = []

patient_terms = ["woman", "man", "girl", "boy", "male", "female", "patient"]
patient_matcher = Matcher(nlp_med.vocab)

pattern_patient = [
    {"IS_DIGIT": True, "OP": "?"},
    {"TEXT": "-", "OP": "?"},
    {"LOWER": "year", "OP": "?"},
    {"TEXT": "-", "OP": "?"},
    {"LOWER": "old", "OP": "?"},
    {"LOWER": {"IN": patient_terms}}
]

patient_matcher.add("PATIENT_DEMO", [pattern_patient])

patient_matches = patient_matcher(doc)

patient_label = "patient"
p_start, p_end, p_start_char, p_end_char = 0, 0, 0, 0

if patient_matches:
    menor_match = min(patient_matches[:5], key=lambda x: x[2] - x[1])
    _, start, end = menor_match

    span = doc[start:end]
    patient_label = span.text.lower()
    p_start, p_end = start, end
    p_start_char, p_end_char = span.start_char, span.end_char

extracted_entities.append({
    'node_type': 'Patient',
    'label_original': span.text if patient_matches else patient_label,
    'label_normalizado': patient_label,
    'token_start': p_start, 'token_end': p_end,
    'span_start': p_start_char, 'span_end': p_end_char,
    'value': None, 'unit': None
})

# Guardamos o nome exato gerado para usar na tabela de arestas depois!
PATIENT_NODE_LABEL = patient_label

### scispaCy para extrair doenças e medicamentos

In [76]:
scispacy_spans = []
for ent in doc.ents:
    scispacy_spans.append(range(ent.start, ent.end))
    extracted_entities.append({
        'node_type': ent.label_, # DISEASE ou CHEMICAL
        'label_original': ent.text,
        'label_normalizado': ent.lemma_.lower(),
        'token_start': ent.start, 'token_end': ent.end,
        'span_start': ent.start_char, 'span_end': ent.end_char,
        'value': None, 'unit': None
    })

entities_df = pd.DataFrame(extracted_entities)
display(entities_df.head())

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,Patient,woman,woman,2,3,14,19,None,None
1,DISEASE,abdominal pain,abdominal pain,14,16,85,99,None,None
2,DISEASE,nausea,nausea,18,19,116,122,None,None
3,DISEASE,constipation,constipation,20,21,127,139,None,None
4,DISEASE,malignancy,malignancy,111,112,664,674,None,None


### Matcher para resgatar exames, procedimentos e conceitos clínicos

In [77]:
matcher = Matcher(nlp_med.vocab)
pattern_exam = [{"POS": {"IN": ["ADJ", "NOUN"]}}, {"POS": "NOUN"}]
matcher.add("CLINICAL_CONCEPT", [pattern_exam])

for match_id, start, end in matcher(doc):
    if not any(start in span or (end-1) in span for span in scispacy_spans):
        span_text = doc[start:end]

        if PATIENT_NODE_LABEL.lower() in span_text.text.lower():
            continue

        # heurística para classificar como exame ou procedimento
        node_class = 'Procedure/Exam' if any(word in span_text.text.lower() for word in ['tomography', 'endoscopy', 'resection', 'pancreatectomy', 'fna', 'examination']) else 'MedicalConcept'

        extracted_entities.append({
            'node_type': node_class,
            'label_original': span_text.text,
            'label_normalizado': span_text.lemma_.lower(),
            'token_start': start, 'token_end': end,
            'span_start': span_text.start_char, 'span_end': span_text.end_char,
            'value': None, 'unit': None
        })

### Tabela de Nós Final

In [78]:
entities_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)
final_nodes_df = pd.concat([entities_df, measurements_df], ignore_index=True)
print(f"TABELA DE NÓS")
display(final_nodes_df.head(15))

TABELA DE NÓS


,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,Patient,woman,woman,2,3,14,19,None,None
1,DISEASE,abdominal pain,abdominal pain,14,16,85,99,None,None
2,DISEASE,nausea,nausea,18,19,116,122,None,None
3,DISEASE,constipation,constipation,20,21,127,139,None,None
4,DISEASE,malignancy,malignancy,111,112,664,674,None,None
5,DISEASE,mucinous pancreatic cystic neoplasm,mucinous pancreatic cystic neoplasm,151,155,875,910,None,None
6,DISEASE,bleeding,bleeding,343,344,1990,1998,None,None
7,CHEMICAL,GDC,gdc,376,377,2148,2151,None,None
8,DISEASE,pain,pain,414,415,2337,2341,None,None
9,MedicalConcept,3-day history,3-day history,6,8,37,50,None,None


## 4. Geração do Grafo de Conhecimento (Tabela de Arestas)

In [79]:
edges = []
edge_id_counter = 1

verb_relations = {
    ("present", "have", "experience", "associate"): "HAS_SYMPTOM",
    ("undergo", "perform", "do", "plan", "convert"): "UNDERWENT_EXAM",
    ("reveal", "demonstrate", "suggest", "show", "note", "observe"): "SUPPORTS",
    ("treat", "resect", "discharge"): "TREATED_BY"
}

### Arestas Sintáticas

In [80]:
for sent in doc.sents:
    # pega apenas os nós que estão nesta frase
    nodes_in_sent = [row for _, row in entities_df.iterrows() if row['node_type'] != 'Patient' and sent.start <= row['token_start'] < sent.end]

    for token in sent:
        if token.pos_ == "VERB":
            verbo_lema = token.lemma_.lower()

            for verbs, rel in verb_relations.items():
                if verbo_lema in verbs:
                    if rel in ["HAS_SYMPTOM", "UNDERWENT_EXAM", "TREATED_BY"]:
                        for node in nodes_in_sent:
                            # filtra um pouco
                            if rel == "HAS_SYMPTOM" and node['node_type'] not in ['DISEASE', 'MedicalConcept']: continue
                            if rel == "UNDERWENT_EXAM" and node['node_type'] not in ['Procedure/Exam']: continue

                            edges.append({
                                'source_label': PATIENT_NODE_LABEL,
                                'target_label': node['label_normalizado'],
                                'relation': rel
                            })

                    elif rel == "SUPPORTS" and len(nodes_in_sent) >= 2:
                        edges.append({
                            'source_label': nodes_in_sent[0]['label_normalizado'],
                            'target_label': nodes_in_sent[1]['label_normalizado'],
                            'relation': rel
                        })
                    break

edges_df = pd.DataFrame(edges)
display(edges_df.head())

,source_label,target_label,relation
0,woman,abdominal pain,HAS_SYMPTOM
1,woman,nausea,HAS_SYMPTOM
2,woman,constipation,HAS_SYMPTOM
3,woman,3-day history,HAS_SYMPTOM
4,woman,right flank,HAS_SYMPTOM


### Arestas de Valores por distância de caracteres

In [81]:
for _, ent_row in entities_df.iterrows():
    if ent_row['node_type'] == 'Patient': continue
    for _, meas_row in measurements_df.iterrows():
        if abs(ent_row['span_start'] - meas_row['span_start']) < 40:
            edges.append({
                'source_label': ent_row['label_normalizado'],
                'target_label': meas_row['label_normalizado'],
                'relation': 'HAS_VALUE'
            })

final_graph_edges_df = pd.DataFrame(edges).drop_duplicates(subset=['source_label', 'target_label', 'relation']).reset_index(drop=True)

## Grafo de Conhecimento final

In [82]:
final_graph_edges_df.insert(0, 'edge_id', [f"E_{i+1:03d}" for i in range(len(final_graph_edges_df))])

display(final_graph_edges_df.head(20))

,edge_id,source_label,target_label,relation
0,E_001,woman,abdominal pain,HAS_SYMPTOM
1,E_002,woman,nausea,HAS_SYMPTOM
2,E_003,woman,constipation,HAS_SYMPTOM
3,E_004,woman,3-day history,HAS_SYMPTOM
4,E_005,woman,right flank,HAS_SYMPTOM
5,E_006,pancreatic echotexture,internal septation,SUPPORTS
6,E_007,woman,pancreatic echotexture,HAS_SYMPTOM
7,E_008,woman,internal septation,HAS_SYMPTOM
8,E_009,malignancy,mucinous pancreatic cystic neoplasm,SUPPORTS
9,E_010,woman,distal pancreatectomy,UNDERWENT_EXAM
